# 6. Streaming with DMA: near the roof

The line-buffer kernel computes a pixel every clock but starves on a byte-wide memory
port. The fix is to change *how* it talks to DRAM, not *what* it computes:

- **Burst** the transfers - ask DRAM for many bytes at once, amortising the access
  latency (DRAM is fast in bulk, slow per byte).
- **Dataflow** - split the kernel into three stages (read -> compute -> write) connected
  by FIFOs, running concurrently. While one row computes, the next is already being read
  and the previous is being written. Memory traffic hides behind compute.

Same line-buffer compute as before, wrapped in a Vitis `DATAFLOW` read/compute/write
pipeline (`hls/conv3x3_stream.cpp`). The csynth reports all three loops at II=1 with
inferred bursts, and it sustains close to one pixel per clock.

> **Board only.** The rungs below need the PYNQ board (`pynq` + the bitstream). On a
> laptop `available_backends()` omits them and the cell prints a skip message - that is
> expected; run this one on the board.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from fpga_conv import KERNELS, get_kernel, conv_reference, to_grayscale_u8, available_backends
from fpga_conv.bench import (
    Scoreboard, run_rung, time_backend, sample_gray, make_sample_image, roofline,
)

# Path to the combined bitstream (carries all four accelerators). Its .hwh must sit
# beside it. Relative to this notebook on the dev box; on the board point it at wherever
# you copied the build, e.g. '/home/xilinx/workshop/conv3x3.bit'.
BIT_PATH = os.path.abspath('../../rtl/build/conv3x3.bit')
HW_KW = {'bitfile': BIT_PATH}          # passed to the hardware rungs only

# The scoreboard persists across the whole series (notebooks/scoreboard.json), so the
# roofline grows as you climb. The hardware rungs are skipped off the board.
sb = Scoreboard()
print('backends available here:', available_backends())

In [ ]:
if 'hls_stream' in available_backends():
    img = sample_gray(512)
    out, t = run_rung('hls_stream', img, 'edges', repeats=10, scoreboard=sb, **HW_KW)
else:
    print('hls_stream backend not available here (laptop) - run this cell on the board.')

This is the top rung: ~40 Mpix/s on the board, counting both the read and the write -
close to what this single byte-wide port and 100 MHz fabric can sustain. The dot now sits
up near the corner where the memory and compute ceilings meet (the *ridge point*): we are
no longer obviously wasting either resource. To go further you would widen the datapath
(process several pixels per clock) or use a wider/parallel memory interface - real, but a
different design.

## The whole climb

Same operation, seven ways. Watch the orders of magnitude.

In [ ]:
sb.roofline(); plt.show()
sb.progress(); plt.show()

## What we learned

- **Beat the easy wins first.** Python loop -> NumPy was the biggest single jump, and it
  was just better software.
- **Hardware wins on parallelism** - nine MACs every clock - but only if you can *feed*
  it. Every rung after the first was a *data-movement* fix, not a compute fix: MMIO ->
  bulk DMA -> read-once line buffer -> bursting dataflow.
- **The bottleneck moves.** Fix compute and memory becomes the wall; fix memory access
  and bandwidth becomes the wall. The roofline tells you which wall you are against.

### The data-science hook

The `edges` kernel you ran is a fixed 3x3 filter. A CNN *learns* its kernels, but the
hardware operation is identical - this accelerator is one convolutional layer. Stack
these with nonlinearities and pooling and you have the inner loop of every modern vision
model, running on the fabric.

*(Bonus: `07_clock_ramp.ipynb` pushes the clock; `99_interactive.ipynb` lets you filter
your own photos.)*